In [10]:
import pandas as pd

# CSV 로드
df = pd.read_csv('/Users/jsh/Desktop/class/3-1/비정형데이터분석/실습/A_DeviceMotion_data 복사본/HAR_summary.csv')

# y(정답), X(입력) 분리
y = df['activity'].astype('category')
X = df.drop(columns=['activity'])

# 데이터 크기/컬럼 확인
print(f"X Shape: {X.shape}, y Shape: {y.shape}")
print(X.columns[:10])

X Shape: (360, 12), y Shape: (360,)
Index(['id', 'exp_no', 'maguserAcceleration_mean', 'maguserAcceleration_min',
       'maguserAcceleration_max', 'maguserAcceleration_std',
       'maguserAcceleration_skew', 'magrotationRate_mean',
       'magrotationRate_min', 'magrotationRate_max'],
      dtype='object')


In [ ]:
from sklearn.preprocessing import LabelEncoder

# 숫자형 컬럼만 사용
X = X.select_dtypes(include=['number'])

# 결측치 0으로 처리
X = X.fillna(0)

# y 라벨 숫자로 변환
le = LabelEncoder()
y_enc = le.fit_transform(y)

# 전처리 결과 확인
print(f"결측치 수: {X.isna().sum().sum()}")
print(f"Classes: {le.classes_}")


결측치 수: 0
Classes: ['dws' 'jog' 'sit' 'std' 'ups' 'wlk']


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# 의사결정나무 모델 생성
dt = DecisionTreeClassifier(max_depth=None, random_state=42)

# 모델 학습
dt.fit(X, y_enc)

# 모델 설정 확인
print("Model Parameters:")
print(dt.get_params())

# 특성 중요도 확인(필요 시)
# print(dt.feature_importances_


Model Parameters:
{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'random_state': 42, 'splitter': 'best'}


In [11]:
from sklearn.ensemble import RandomForestClassifier

# 랜덤포레스트 모델 생성
rf = RandomForestClassifier(
 n_estimators=200,
 max_depth=None,
 n_jobs=-1,
 random_state=42
)

# 모델 학습
rf.fit(X, y_enc)

# 모델 설정 확인
print(rf.get_params())


{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 200, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# 10-Fold 교차검증 설정
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 교차검증 예측
y_pred = cross_val_predict(rf, X, y_enc, cv=cv, n_jobs=-1)

# 예측 결과 확인
print(f"Predictions shape: {y_pred.shape}")
print(y_pred[:10])


Predictions shape: (360,)
[0 4 4 2 3 5 1 0 4 4]


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

# 분류 리포트 출력
print("=== Classification Report ===")
print(classification_report(y_enc, y_pred))

# 혼동행렬 출력
print("=== Confusion Matrix ===")
print(confusion_matrix(y_enc, y_pred))


=== Classification Report ===
              precision    recall  f1-score   support

           0       1.00      0.88      0.93        72
           1       0.94      0.94      0.94        48
           2       0.92      0.92      0.92        48
           3       0.92      0.92      0.92        48
           4       0.93      0.97      0.95        72
           5       0.88      0.96      0.92        72

    accuracy                           0.93       360
   macro avg       0.93      0.93      0.93       360
weighted avg       0.93      0.93      0.93       360

=== Confusion Matrix ===
[[63  1  0  0  4  4]
 [ 0 45  0  0  0  3]
 [ 0  0 44  4  0  0]
 [ 0  0  4 44  0  0]
 [ 0  0  0  0 70  2]
 [ 0  2  0  0  1 69]]


In [14]:
import joblib

# 모델 저장
joblib.dump(rf, 'HAR04_model.pkl')
print("Model saved successfully!")

# 모델 불러오기
rf_loaded = joblib.load('HAR04_model.pkl')

# 불러온 모델 확인
print(f"Type: {type(rf_loaded)}")
print(rf_loaded.get_params())

Model saved successfully!
Type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 200, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [15]:
# 연습문제
# 1. DecisionTree와 RandomForest 모델을 각각 학습시키고, 10-fold 교차검증을 통해 성능 차이를 분석하세요. 

y = df['activity'].astype('category')
X = df.drop(columns=['activity'], errors='ignore')

# 숫자형 + 결측치 처리
X = X.select_dtypes(include=['number']).fillna(0)

# 라벨 인코딩
le = LabelEncoder()
y_enc = le.fit_transform(y)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "DecisionTree": DecisionTreeClassifier(max_depth=None, random_state=42),
    "RandomForest_base": RandomForestClassifier(
        n_estimators=200, max_depth=None, n_jobs=-1, random_state=42
    ),
    "RandomForest_tuned": RandomForestClassifier(
        n_estimators=400, max_depth=20, min_samples_split=4, n_jobs=-1, random_state=42
    ),
}

for name, model in models.items():
    y_pred = cross_val_predict(model, X, y_enc, cv=cv, n_jobs=-1)

    print(f"\n=== {name} ===")
    print(classification_report(y_enc, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_enc, y_pred))



=== DecisionTree ===
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.96      0.94      0.95        48
           2       0.91      0.90      0.91        48
           3       0.90      0.92      0.91        48
           4       1.00      0.99      0.99        72
           5       0.97      0.97      0.97        72

    accuracy                           0.96       360
   macro avg       0.95      0.95      0.95       360
weighted avg       0.96      0.96      0.96       360

Confusion Matrix:
[[71  1  0  0  0  0]
 [ 2 45  0  0  0  1]
 [ 0  0 43  5  0  0]
 [ 0  0  4 44  0  0]
 [ 0  0  0  0 71  1]
 [ 1  1  0  0  0 70]]

=== RandomForest_base ===
              precision    recall  f1-score   support

           0       1.00      0.88      0.93        72
           1       0.94      0.94      0.94        48
           2       0.92      0.92      0.92        48
           3       0.92      0.92      0.92   

In [ ]:
# 연습문제 
# 2. 제공된 데이터셋(HAR_summary.csv)에 있는 동일한 특질(X)을 사용하여 'gender' 컬럼을 예측하는 새로운 분류 모델을 만드세요. 


# 파일 경로
HAR_PATH = '/Users/jsh/Desktop/class/3-1/비정형데이터분석/실습/A_DeviceMotion_data 복사본/HAR_summary.csv'
SUBJECT_PATH = '/Users/jsh/Desktop/class/3-1/비정형데이터분석/motion-sense-master/data/data_subjects_info.csv'

# 1) 데이터 로드
har = pd.read_csv(HAR_PATH)
sub = pd.read_csv(SUBJECT_PATH)

# 컬럼 공백 제거
har.columns = har.columns.str.strip()
sub.columns = sub.columns.str.strip()

# 2) gender 붙이기 (id <-> code)
df = har.merge(
    sub[['code', 'gender']],
    left_on='id',
    right_on='code',
    how='left'
)

# 불필요 컬럼 제거
df = df.drop(columns=['code'], errors='ignore')

# 3) 타깃/입력 분리
if 'gender' not in df.columns:
    raise ValueError("gender 컬럼 생성 실패: merge 키(id/code) 확인 필요")

y = df['gender'].astype('category')

# 기존 activity 예측 특질을 최대한 유지하되 타깃 컬럼 제거
X = df.drop(columns=['gender'], errors='ignore')

# 숫자형만 + 결측치 처리
X = X.select_dtypes(include=['number']).fillna(0)

# 4) 라벨 인코딩
le = LabelEncoder()
y_enc = le.fit_transform(y)

# 5) 학습/평가 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# 6) 모델 학습 (RandomForest 권장)
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 7) 성능 확인
y_pred = model.predict(X_test)
print("=== Classification Report ===")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_.astype(str)
))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

# 8) 모델 저장
joblib.dump(model, 'gender_model.pkl')
print("Saved: gender_model.pkl")

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.91      0.70      0.79        30
           1       0.82      0.95      0.88        42

    accuracy                           0.85        72
   macro avg       0.86      0.83      0.84        72
weighted avg       0.86      0.85      0.84        72

=== Confusion Matrix ===
[[21  9]
 [ 2 40]]
Saved: gender_model.pkl
